# EDA — Marche_Principal_Actions_MERGED.csv

In [4]:
import pandas as pd
import numpy as np

df_actions = pd.read_csv('D:/rouge_file/Analyse_du_performance_de_la_bourse_casablanca/Merged/Marche_Principal_Actions_MERGED.csv', low_memory=False)

In [5]:
print(f'Shape : {df_actions.shape}')

Shape : (52070, 22)


In [6]:
df_actions.head()

,Type,Capital en titres,Val. Nom.,Secteur activité,Cycle de négociation,Dernier Dividende ajusté,Unnamed: 6,Unnamed: 7,Cours de référence,Unnamed: 9,...,Unnamed: 12,Cours du jour,Unnamed: 14,Unnamed: 15,Unnamed: 16,Statut,Meilleurs cours à la clôture,Unnamed: 19,**Depuis janvier,Unnamed: 21
0,Principal A,3 437 500,100,PEG,C,"140,00",2021,05/05/22,"4 600,00",03/10/22,...,AFRIQUIA GAZ,-,"4 600,00",-,-,NT,"4 501,00","4 735,00","5 950,00","4 232,00"
1,Principal A,60 283 595,10,ASS,C,"5,20",2021,27/05/22,"127,00",03/10/22,...,ATLANTASANAD,"125,00","128,00","128,00","125,00",T,"125,05","128,00","145,00","112,00"
2,Principal A,215 140 839,10,BAN,C,"15,00",2021,05/07/22,"425,00",03/10/22,...,ATTUARIWAFA BANK,"421,00","415,20","422,00","415,20",T,"415,00","430,00","498,95","410,00"
3,Principal A,50 294 528,10,DIS,C,"3,50",2021,09/08/22,"73,50",03/10/22,...,AUTO HALL,"74,00","75,00","75,00","74,00",T,"74,00","77,79","114,80","73,50"
4,Principal A,205 606 648,10,BAN,C,"3,94",2021,14/07/22,"161,00",03/10/22,...,BANK OF AFRICA,"165,10","165,30","165,30","165,00",T,"165,00","173,00","221,56","161,00"


## 1. Qualite des donnees

In [7]:
print('Valeurs manquantes :')
print(df_actions.isnull().sum())
print(f'\nDoublons         : {df_actions.duplicated().sum()}')
print(f'Lignes parasites : {(df_actions["Type"] == "Type").sum()}')
print('\nValeurs colonne Type :')
print(df_actions['Type'].value_counts())

Valeurs manquantes :
Type                               0
Capital en titres                 90
Val. Nom.                         99
Secteur activité                 108
Cycle de négociation             108
Dernier Dividende ajusté        1996
Unnamed: 6                       819
Unnamed: 7                      1996
Cours de référence               123
Unnamed: 9                       123
* Cours de référence séance      123
Instruments                      123
Unnamed: 12                     1297
Cours du jour                   4595
Unnamed: 14                      124
Unnamed: 15                      124
Unnamed: 16                      124
Statut                           132
Meilleurs cours à la clôture     215
Unnamed: 19                      135
**Depuis janvier                 250
Unnamed: 21                      476
dtype: int64

Doublons         : 5423
Lignes parasites : 33

Valeurs colonne Type :
Type
Principal B     30351
Principal A     15944
Alternatif A     3393
Principal 

## 2. Nettoyage

In [8]:
# Suppression parasites + doublons
df_actions = df_actions[df_actions['Type'] != 'Type'].drop_duplicates().copy()

# Renommage des colonnes Unnamed
df_actions = df_actions.rename(columns={
    'Unnamed: 6'  : 'Annee_Dividende',
    'Unnamed: 7'  : 'Date_Dividende',
    'Unnamed: 9'  : 'Date_Cours_Ref',
    'Unnamed: 12' : 'Libelle',
    'Unnamed: 14' : 'Cours_Cloture',
    'Unnamed: 15' : 'Cours_Haut',
    'Unnamed: 16' : 'Cours_Bas',
    'Unnamed: 19' : 'Meilleur_Offre',
    'Unnamed: 21' : 'Plus_Bas_Janv',
    '**Depuis janvier' : 'Plus_Haut_Janv',
    'Meilleurs cours a la cloture' : 'Meilleur_Demande',
})

# Nettoyage colonne Statut (valeurs parasites)
statuts_valides = ['T', 'NT', 'S', '-', 'Banc']
df_actions['Statut'] = df_actions['Statut'].where(df_actions['Statut'].isin(statuts_valides), other=np.nan)

# Conversion numerique
def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace(' ','').str.replace('\xa0','')
         .str.replace(',','.').str.replace('-','').str.strip(),
        errors='coerce'
    )

cols_num = ['Capital en titres', 'Val. Nom.', 'Dernier Dividende ajusté',
            'Cours de référence', '* Cours de référence séance',
            'Cours du jour', 'Cours_Cloture', 'Cours_Haut', 'Cours_Bas',
            'Meilleur_Offre', 'Meilleur_Demande', 'Plus_Haut_Janv', 'Plus_Bas_Janv']

for col in cols_num:
    if col in df_actions.columns:
        df_actions[col] = to_num(df_actions[col])

# Conversion dates
for col in ['Date_Dividende', 'Date_Cours_Ref']:
    if col in df_actions.columns:
        df_actions[col] = pd.to_datetime(df_actions[col], format='%d/%m/%y', errors='coerce')

print(f'Shape apres nettoyage : {df_actions.shape}')
df_actions.head(3)

Shape apres nettoyage : (46622, 22)


,Type,Capital en titres,Val. Nom.,Secteur activité,Cycle de négociation,Dernier Dividende ajusté,Annee_Dividende,Date_Dividende,Cours de référence,Date_Cours_Ref,...,Libelle,Cours du jour,Cours_Cloture,Cours_Haut,Cours_Bas,Statut,Meilleurs cours à la clôture,Meilleur_Offre,Plus_Haut_Janv,Plus_Bas_Janv
0,Principal A,3437500.0,100.0,PEG,C,140.0,2021,2022-05-05,4600.0,2022-10-03,...,AFRIQUIA GAZ,NaN,4600.0,NaN,NaN,NT,"4 501,00",4735.0,5950.00,4232.0
1,Principal A,60283595.0,10.0,ASS,C,5.2,2021,2022-05-27,127.0,2022-10-03,...,ATLANTASANAD,125.0,128.0,128.0,125.0,T,"125,05",128.0,145.00,112.0
2,Principal A,215140839.0,10.0,BAN,C,15.0,2021,2022-07-05,425.0,2022-10-03,...,ATTUARIWAFA BANK,421.0,415.2,422.0,415.2,T,"415,00",430.0,498.95,410.0


## 3. Statistiques descriptives

In [9]:
df_actions[['Cours de référence','* Cours de référence séance',
    'Cours du jour','Cours_Cloture','Cours_Haut','Cours_Bas']].describe().round(2)

,Cours de référence,* Cours de référence séance,Cours du jour,Cours_Cloture,Cours_Haut,Cours_Bas
count,46594.00,46594.00,37038.00,45404.00,41196.00,40419.00
mean,929.99,930.55,904.22,921.12,897.57,867.94
std,1268.07,1268.76,1188.82,1238.38,1240.91,1183.37
min,5.65,5.65,5.70,5.65,5.67,5.64
25%,153.00,153.00,154.01,154.85,134.78,132.70
50%,450.00,450.00,448.20,450.00,425.00,413.10
75%,1187.00,1188.75,1200.00,1193.00,1130.00,1107.50
max,8700.00,8700.00,8460.00,8700.00,8797.00,8401.00


In [10]:
# Stats par compartiment (Type)
df_actions.groupby('Type')[['Cours de référence','Cours du jour','Capital en titres']].mean().round(2)

,Cours de référence,Cours du jour,Capital en titres
Type,,,
Alternatif A,1273.85,783.70,4097400.75
Non spécifié,935.23,761.65,56926253.74
Principal A,1397.81,1353.04,95158345.59
Principal B,632.96,602.19,11298548.35
Principal F,404.20,457.49,25579711.64


## 4. Analyse par Secteur

In [11]:
print('Nombre de societes par secteur :')
print(df_actions.groupby('Secteur activité')['Libelle'].nunique().sort_values(ascending=False).to_string())

Nombre de societes par secteur :
Secteur activité
BAN    16
SPH    15
BMC    14
DIS    12
ASS    10
LSI     9
AGR     7
IBE     7
SFA     7
MIN     5
PHA     4
SDT     3
SPI     3
PEG     3
IMM     3
BOI     2
TRP     2
SAN     2
CHI     2
IAG     1
ELC     1
BCI     1
LH      1
SAC     1
SF      1
SP      1
TCO     1


In [12]:
perf_secteur = (
    df_actions.groupby('Secteur activité')
    .agg(
        Cours_Moyen      = ('Cours de référence', 'mean'),
        Cours_Min        = ('Cours de référence', 'min'),
        Cours_Max        = ('Cours de référence', 'max'),
        Capital_Moyen    = ('Capital en titres',  'mean'),
        Nb_observations  = ('Cours de référence', 'count')
    )
    .round(2)
    .sort_values('Cours_Moyen', ascending=False)
)
print('Performance par secteur (cours moyen) :')
print(perf_secteur.to_string())

Performance par secteur (cours moyen) :
                  Cours_Moyen  Cours_Min  Cours_Max  Capital_Moyen  Nb_observations
Secteur activité                                                                   
PEG                   2784.73     127.80    4847.00   6.257078e+06             1350
ASS                   2659.43     118.00    7500.00   1.512843e+07             3071
BCI                   2390.00    2180.00    2600.00   2.829653e+06                2
MIN                   1995.39      65.71    8700.00   3.965181e+06             2499
BOI                   1963.45    1150.00    2900.00   2.494534e+06             1075
ELC                   1646.22     940.00    3164.00   2.358854e+07              677
PHA                   1162.30     873.00    1949.00   4.431768e+06             1221
BMC                   1074.64      41.10    2850.00   1.206658e+07             5439
DIS                    965.97      10.23    5150.00   1.559803e+07             4287
SAN                    842.05     23

## 5. Analyse par Societe

In [13]:
print(f'Nombre de societes distinctes : {df_actions["Libelle"].nunique()}')
print()
print('Top 10 cours moyen le plus eleve :')
top10 = (
    df_actions.groupby('Libelle')['Cours de référence']
    .mean().round(2)
    .sort_values(ascending=False)
    .head(10)
)
print(top10.to_string())
print()
print('Top 10 plus fort capital en titres :')
top_cap = (
    df_actions.groupby('Libelle')['Capital en titres']
    .mean().round(0)
    .sort_values(ascending=False)
    .head(10)
)
print(top_cap.to_string())

Nombre de societes distinctes : 133

Top 10 cours moyen le plus eleve :
Libelle
AGMA               6682.76
APS                6200.00
WAFA ASSURANCE     4587.34
LABEL VIE          4448.70
WFA ASSURANCE      4232.15
WFAA ASSURANCE     4214.29
WAFFA ASSURANCE    4170.54
AFRIQUIA GAZ       4133.82
WFAFA ASSURANCE    4100.00
WAF A ASSURANCE    4092.00

Top 10 plus fort capital en titres :
Libelle
ITISSALAT AL-MAGHRIB    879095340.0
DOUJA PROM ADDOHA       402551254.0
ATTUARIWAFFA BANK       215140839.0
ATTUARIWAFA BANK        215140839.0
ATTUARIWAF A BANK       215140839.0
ATTUARIWFA BANK         215140839.0
ATTIJARIWAFA BANK       215140839.0
BANK OF AFRICA          213839384.0
BCP                     203312473.0
BOP                     203312473.0


## 6. Analyse du Statut de Cotation

In [14]:
print('Distribution des statuts :')
statut_count = df_actions['Statut'].value_counts()
statut_pct   = (statut_count / statut_count.sum() * 100).round(2)
print(pd.DataFrame({'Nb': statut_count, '%': statut_pct}).to_string())
print()
print('Legende : T=Traite | NT=Non Traite | S=Suspendu | Banc=Bande de cours')

Distribution des statuts :
           Nb      %
Statut              
T       37030  85.59
NT       5254  12.14
-         917   2.12
S          61   0.14
Banc        4   0.01

Legende : T=Traite | NT=Non Traite | S=Suspendu | Banc=Bande de cours


## 7. Analyse des Dividendes

In [15]:
div = df_actions[df_actions['Dernier Dividende ajusté'] > 0]
print(f'Societes ayant distribue un dividende : {div["Libelle"].nunique()}')
print()
print('Top 10 dividendes les plus eleves :')
top_div = (
    div.groupby('Libelle')['Dernier Dividende ajusté']
    .max().round(2)
    .sort_values(ascending=False)
    .head(10)
)
print(top_div.to_string())

Societes ayant distribue un dividende : 127

Top 10 dividendes les plus eleves :
Libelle
AGMA                             300.0
AFRIQUIA GAZ                     175.0
SOCIETE DES BOISSONS DU MAROC    160.0
WFAA ASSURANCE                   140.0
WAFA ASSURANCE                   140.0
WAFFA ASSURANCE                  140.0
DARI COUSPATE                    140.0
WAF A ASSURANCE                  140.0
WFA ASSURANCE                    140.0
WFAFA ASSURANCE                  130.0


## 8. Outliers

In [16]:
for col in ['Cours de référence', 'Cours du jour', 'Capital en titres']:
    Q1, Q3 = df_actions[col].quantile(0.25), df_actions[col].quantile(0.75)
    IQR    = Q3 - Q1
    out    = df_actions[(df_actions[col] < Q1-1.5*IQR) | (df_actions[col] > Q3+1.5*IQR)]
    print(f'{col:35s} : {len(out):,} outliers ({len(out)/len(df_actions)*100:.1f}%)')

Cours de référence                  : 3,568 outliers (7.7%)
Cours du jour                       : 2,562 outliers (5.5%)
Capital en titres                   : 6,143 outliers (13.2%)


## 9. Correlation

In [17]:
df_actions[['Cours de référence','* Cours de référence séance',
    'Cours du jour','Cours_Haut','Cours_Bas',
    'Capital en titres','Dernier Dividende ajusté']].corr().round(4)

,Cours de référence,* Cours de référence séance,Cours du jour,Cours_Haut,Cours_Bas,Capital en titres,Dernier Dividende ajusté
Cours de référence,1.0000,0.9994,0.9996,0.9996,0.9996,-0.1722,0.8583
* Cours de référence séance,0.9994,1.0000,0.9994,0.9996,0.9995,-0.1722,0.8580
Cours du jour,0.9996,0.9994,1.0000,0.9998,0.9998,-0.1948,0.8170
Cours_Haut,0.9996,0.9996,0.9998,1.0000,0.9997,-0.1777,0.8428
Cours_Bas,0.9996,0.9995,0.9998,0.9997,1.0000,-0.1809,0.8331
Capital en titres,-0.1722,-0.1722,-0.1948,-0.1777,-0.1809,1.0000,-0.1774
Dernier Dividende ajusté,0.8583,0.8580,0.8170,0.8428,0.8331,-0.1774,1.0000


## 10. Export

In [18]:


print('Colonnes avant renommage:')
print(list(df_actions.columns))

df_actions = df_actions.rename(columns={
    'Unnamed: 6'               : 'Annee_Dividende',
    'Unnamed: 7'               : 'Date_Dividende',
    'Unnamed: 9'               : 'Date_Cours_Reference',
    'Unnamed: 12'              : 'Libelle',
    'Unnamed: 14'              : 'Cours_Cloture',
    'Unnamed: 15'              : 'Cours_Haut_Seance',
    'Unnamed: 16'              : 'Cours_Bas_Seance',
    'Unnamed: 19'              : 'Meilleur_Offre',
    '**Depuis janvier'         : 'Plus_Haut_Annee',
    'Unnamed: 21'              : 'Plus_Bas_Annee',
    'Meilleurs cours à la clôture' : 'Meilleur_Demande',
    '* Cours de référence séance'  : 'Cours_Reference_Seance',
})

print()
print('Colonnes apres renommage:')
print(list(df_actions.columns))

df_actions.to_csv('Actions_RENAMED.csv', index=False, encoding='utf-8')
print()
print(f'Exporte : Actions_RENAMED.csv — {len(df_actions):,} lignes')

Colonnes avant renommage:
['Type', 'Capital en titres', 'Val. Nom.', 'Secteur activité', 'Cycle de négociation', 'Dernier Dividende ajusté', 'Annee_Dividende', 'Date_Dividende', 'Cours de référence', 'Date_Cours_Ref', '* Cours de référence séance', 'Instruments', 'Libelle', 'Cours du jour', 'Cours_Cloture', 'Cours_Haut', 'Cours_Bas', 'Statut', 'Meilleurs cours à la clôture', 'Meilleur_Offre', 'Plus_Haut_Janv', 'Plus_Bas_Janv']

Colonnes apres renommage:
['Type', 'Capital en titres', 'Val. Nom.', 'Secteur activité', 'Cycle de négociation', 'Dernier Dividende ajusté', 'Annee_Dividende', 'Date_Dividende', 'Cours de référence', 'Date_Cours_Ref', 'Cours_Reference_Seance', 'Instruments', 'Libelle', 'Cours du jour', 'Cours_Cloture', 'Cours_Haut', 'Cours_Bas', 'Statut', 'Meilleur_Demande', 'Meilleur_Offre', 'Plus_Haut_Janv', 'Plus_Bas_Janv']

Exporte : Actions_RENAMED.csv — 46,622 lignes
